# Clip Extraction — YOLO11-pose + BoT-SORT
### Perubahan dari versi sebelumnya:
| Aspek | YOLOv8 + HRNet (lama) | YOLO11-pose (baru) |
|---|---|---|
| Deteksi manusia | `YOLO('yolov8m.pt')` | `YOLO('yolo11m-pose.pt')` |
| Pose estimation | `pose_inference()` via MMPose | Built-in di YOLO11-pose |
| Jumlah model | 2 model terpisah | 1 model saja |
| Output keypoint | MMPose `PoseDataSample` | `results.keypoints` |
| Visualisasi skeleton | MMPose `VISUALIZERS` | OpenCV manual |
| `human_detections` | Dibutuhkan untuk `pose_inference` | **Tidak diperlukan** |


## 1. Import & Konfigurasi

In [1]:
# Copyright (c) CIIS-Lab. All rights reserved.
import os
import os.path as osp
import copy as cp
import tempfile
import gc
import glob
import csv

import cv2
import mmcv
import mmengine
import numpy as np
import torch

from ultralytics import YOLO
from boxmot import BoTSORT
from pathlib import Path
from mmaction.utils import frame_extract
import moviepy.editor as mpy


In [2]:
FONTFACE  = cv2.FONT_HERSHEY_DUPLEX
FONTSCALE = 1
THICKNESS = 2
LINETYPE  = 1


In [4]:
# ============================================================
#  KONFIGURASI — sesuaikan path sebelum dijalankan
# ============================================================

INPUT_DIR  = '03_dataset/data_baru/lainnya'        # folder berisi semua video .mp4
OUTPUT_DIR = '03_dataset/data_out/2_out'        # folder beda agar tidak overwrite
PKL_DIR    = '03_dataset/data_in/2_csv' # folder .csv + .pkl mentah per video

# YOLO11-pose — satu model untuk deteksi + pose sekaligus
# Pilihan: yolo11n-pose (cepat), yolo11m-pose (seimbang), yolo11x-pose (akurat)
MODEL_POSE_PATH = '01_asset_tools/models/yolo11m-pose.pt'
model_pose      = YOLO(MODEL_POSE_PATH)
det_score_thr   = 0.3

label_map_stdet  = '03_dataset/ciis_label_map.txt'
predict_stepsize = 4      # sama seperti pipeline lama
output_fps       = 12
device           = 'cuda:0'

BOTSORT_REID_WEIGHTS = Path('01_asset_tools/models/osnet_x0_25_msmt17.pt')
BOTSORT_DEVICE       = device
BOTSORT_HALF         = True

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PKL_DIR,    exist_ok=True)
print('Konfigurasi selesai.')
print(f'Input  : {INPUT_DIR}')
print(f'Output : {OUTPUT_DIR}')
print(f'PKL    : {PKL_DIR}')


Konfigurasi selesai.
Input  : 03_dataset/data_baru/lainnya
Output : 03_dataset/data_out/2_out
PKL    : 03_dataset/data_in/2_csv


## 2. Helper Functions

In [5]:
def hex2color(h):
    return (int(h[:2], 16), int(h[2:4], 16), int(h[4:], 16))

PLATEBLUE = [hex2color(h) for h in '03045e-023e8a-0077b6-0096c7-00b4d8-48cae4'.split('-')]

def abbrev(name):
    while name.find('(') != -1:
        st, ed = name.find('('), name.find(')')
        name = name[:st] + '...' + name[ed + 1:]
    return name

def _cal_iou(box1, box2):
    xmin1, ymin1, xmax1, ymax1 = box1
    xmin2, ymin2, xmax2, ymax2 = box2
    s1    = max(0, xmax1 - xmin1) * max(0, ymax1 - ymin1)
    s2    = max(0, xmax2 - xmin2) * max(0, ymax2 - ymin2)
    xi    = max(0, min(xmax1, xmax2) - max(xmin1, xmin2))
    yi    = max(0, min(ymax1, ymax2) - max(ymin1, ymin2))
    inter = xi * yi
    union = s1 + s2 - inter
    return inter / union if union > 0 else 0


## 3. YOLO11-pose + BoT-SORT Tracking

Menggantikan `run_detection_and_tracking` + `pose_inference` dari versi lama.
Output `pose_results` diformat **sama persis** dengan output HRNet agar fungsi
`build_track_pose_index` dan `skeleton_based_stdet_botsort` tidak perlu diubah.


In [6]:
def run_detection_pose_yolo11(frame_paths, model_pose, det_score_thr,
                               reid_weights, track_device, half=True):
    """
    YOLO11-pose + BoT-SORT untuk setiap frame.

    Returns:
        tracked_detections : list[list[dict]]  per frame → {track_id, bbox, score}
        pose_results       : list[dict]        per frame → {keypoints, keypoint_scores, bboxes}
    """
    tracker = BoTSORT(
        model_weights=reid_weights,
        device=track_device,
        fp16=half,
        track_high_thresh=det_score_thr,
        track_low_thresh=0.1,
        new_track_thresh=det_score_thr,
        track_buffer=50,
        match_thresh=0.8,
        proximity_thresh=0.5,
        appearance_thresh=0.25,
        with_reid=True,
    )

    tracked_detections = []
    pose_results       = []

    print('Running YOLO11-pose + BoT-SORT tracking...')
    prog_bar = mmengine.ProgressBar(len(frame_paths))

    for frame_path in frame_paths:
        frame_bgr = cv2.imread(frame_path)

        # YOLO11-pose: deteksi + keypoint sekaligus dalam satu inference
        res = model_pose(frame_bgr, classes=[0], verbose=False)[0]

        # Ambil output dengan pengecekan None
        if res.boxes is not None and len(res.boxes):
            boxes  = res.boxes.xyxy.cpu().numpy()
            scores = res.boxes.conf.cpu().numpy()
        else:
            boxes  = np.zeros((0, 4))
            scores = np.zeros((0,))

        if (res.keypoints is not None
                and res.keypoints.xy is not None
                and len(res.keypoints.xy)):
            kps    = res.keypoints.xy.cpu().numpy()    # [N, 17, 2]
            kpconf = res.keypoints.conf.cpu().numpy()  # [N, 17]
        else:
            kps    = np.zeros((0, 17, 2))
            kpconf = np.zeros((0, 17))

        # Filter confidence
        if len(scores) > 0:
            mask   = scores >= det_score_thr
            boxes  = boxes[mask];  scores = scores[mask]
            kps    = kps[mask];    kpconf = kpconf[mask]

        # BoT-SORT — format input sama seperti versi YOLOv8
        dets = np.hstack([boxes, scores[:, None], np.zeros((len(boxes), 1))])                if len(boxes) > 0 else np.empty((0, 6))
        tracks = tracker.update(dets, frame_bgr)

        # Format tracked_detections — identik dengan versi YOLOv8
        frame_tracks = []
        if len(tracks) > 0:
            for t in tracks:
                x1, y1, x2, y2 = t[0], t[1], t[2], t[3]
                tid   = int(t[4])
                score = float(t[5])
                frame_tracks.append({'track_id': tid,
                                     'bbox': [x1, y1, x2, y2],
                                     'score': score})
        tracked_detections.append(frame_tracks)

        # Format pose_results — identik dengan output pose_inference HRNet
        # Kunci: bboxes diisi dari boxes YOLO11 (sebelum filter track),
        # agar IoU matching di build_track_pose_index bisa berjalan
        pose_results.append({
            'keypoints':       kps,     # [N, 17, 2]
            'keypoint_scores': kpconf,  # [N, 17]
            'bboxes':          boxes,   # [N, 4]
        })

        prog_bar.update()

    print(f'\nSelesai. Total frame: {len(frame_paths)}')
    return tracked_detections, pose_results


## 4. Clip Extraction berbasis Track ID


In [7]:
def build_track_pose_index(tracked_detections, pose_results):
    track_pose_index = []
    for frame_idx, (frame_tracks, frame_poses) in enumerate(
            zip(tracked_detections, pose_results)):
        id_to_pose = {}
        if not frame_tracks or len(frame_poses.get('keypoints', [])) == 0:
            track_pose_index.append(id_to_pose)
            continue
        pose_bboxes = frame_poses['bboxes']
        for track in frame_tracks:
            tid = track['track_id']
            best_iou, best_pidx = -1, -1
            for pidx, pbbox in enumerate(pose_bboxes):
                iou = _cal_iou(track['bbox'], pbbox)
                if iou > best_iou:
                    best_iou, best_pidx = iou, pidx
            if best_pidx >= 0 and best_iou > 0.1:
                id_to_pose[tid] = best_pidx
        track_pose_index.append(id_to_pose)
    return track_pose_index


def skeleton_based_stdet_botsort(predict_stepsize, video,
                                  tracked_detections, pose_results,
                                  num_frame, clip_len, frame_interval, h, w):
    window_size = clip_len * frame_interval
    assert clip_len % 2 == 0, 'clip_len harus genap'

    timestamps = np.arange(
        window_size // 2,
        num_frame + 1 - window_size // 2,
        predict_stepsize
    )

    print('Building track→pose index...')
    track_pose_index = build_track_pose_index(tracked_detections, pose_results)

    skeleton_predictions = []
    skeleton_datasets    = []

    print('Extracting clips per track_id...')
    prog_bar = mmengine.ProgressBar(len(timestamps))

    for timestamp in timestamps:
        start_frame   = timestamp - (clip_len // 2 - 1) * frame_interval
        frame_inds    = list(start_frame + np.arange(0, window_size, frame_interval) - 1)
        frame_inds    = [max(0, min(int(fi), num_frame - 1)) for fi in frame_inds]
        n_clip_frames = len(frame_inds)

        center_idx    = int(timestamp) - 1
        active_tracks = tracked_detections[center_idx]

        if not active_tracks:
            skeleton_predictions.append(None)
            prog_bar.update()
            continue

        skeleton_prediction = []

        for i, track in enumerate(active_tracks):
            tid = track['track_id']
            skeleton_prediction.append([])

            keypoint       = np.zeros((1, n_clip_frames, 17, 2))
            keypoint_score = np.zeros((1, n_clip_frames, 17))

            for j, fi in enumerate(frame_inds):
                pose_idx = track_pose_index[fi].get(tid, None)
                if pose_idx is not None:
                    keypoint[0, j]       = pose_results[fi]['keypoints'][pose_idx]
                    keypoint_score[0, j] = pose_results[fi]['keypoint_scores'][pose_idx]

            frame_dir_name = (
                osp.splitext(osp.basename(video))[0]
                + f'_t{int(timestamp)}_id{tid}'
            )
            csv_id = int(timestamp) + (i + 1) * 0.001

            fake_anno = dict(
                frame_dir=frame_dir_name,
                label=-1,
                img_shape=(h, w),
                original_shape=(h, w),
                num_clips=1,
                total_frames=n_clip_frames,
                keypoint=keypoint,
                keypoint_score=keypoint_score,
                track_id=tid,
                csv_id=csv_id,
            )

            skeleton_datasets.append(fake_anno)
            skeleton_prediction[i].append(('annotate!', csv_id))

        skeleton_predictions.append(skeleton_prediction)
        prog_bar.update()

    return timestamps, skeleton_predictions, skeleton_datasets


## 5. Visualisasi

Skeleton digambar manual via OpenCV — menggantikan MMPose `VISUALIZERS`
karena YOLO11-pose tidak menghasilkan `pose_datasample` format MMPose.
Semua overlay bbox, track ID, dan label CSV tetap sama.


In [8]:
# Koneksi keypoint COCO-17
# Warna per bagian tubuh — mirip seperti MMPose default
SKELETON_CONNECTIONS = [
    # (keypoint_a, keypoint_b, warna_BGR)
    (0, 1,  (255, 128, 0)),    # kepala kiri
    (0, 2,  (0, 128, 255)),    # kepala kanan
    (1, 3,  (255, 128, 0)),    # telinga kiri
    (2, 4,  (0, 128, 255)),    # telinga kanan
    (5, 6,  (0, 255, 0)),      # bahu
    (5, 7,  (255, 128, 0)),    # lengan atas kiri
    (7, 9,  (255, 128, 0)),    # lengan bawah kiri
    (6, 8,  (0, 128, 255)),    # lengan atas kanan
    (8, 10, (0, 128, 255)),    # lengan bawah kanan
    (5, 11, (0, 255, 128)),    # torso kiri
    (6, 12, (0, 255, 128)),    # torso kanan
    (11, 12,(0, 255, 0)),      # pinggul
    (11, 13,(255, 128, 0)),    # paha kiri
    (13, 15,(255, 128, 0)),    # betis kiri
    (12, 14,(0, 128, 255)),    # paha kanan
    (14, 16,(0, 128, 255)),    # betis kanan
]

def draw_skeleton_opencv(frame, keypoints, keypoint_scores, kpt_thr=0.3):
    for pid in range(len(keypoints)):
        kps   = keypoints[pid]
        score = keypoint_scores[pid]

        # Garis dengan warna per bagian tubuh
        for (i, j, color) in SKELETON_CONNECTIONS:
            if score[i] >= kpt_thr and score[j] >= kpt_thr:
                x1, y1 = int(kps[i][0]), int(kps[i][1])
                x2, y2 = int(kps[j][0]), int(kps[j][1])
                if x1 > 0 and y1 > 0 and x2 > 0 and y2 > 0:
                    cv2.line(frame, (x1,y1), (x2,y2), color, 2, cv2.LINE_AA)

        # Titik keypoint
        for k in range(17):
            if score[k] >= kpt_thr:
                x, y = int(kps[k][0]), int(kps[k][1])
                if x > 0 and y > 0:
                    cv2.circle(frame, (x,y), 3, (255,255,255), -1, cv2.LINE_AA)

    return frame

def visualize_yolo11(frames, annotations, pose_results,
                     action_result, plate=PLATEBLUE, max_num=5,
                     tracked_detections=None, output_timestamps=None,
                     all_timestamps=None, predict_stepsize=4):
    frames_    = cp.deepcopy(frames)
    frames_    = [mmcv.imconvert(f, 'bgr', 'rgb') for f in frames_]
    bahaya     = ['melempar', 'membidik senapan', 'membidik pistol',
                  'memukul', 'menendang', 'menusuk']
    anchor_set = set(all_timestamps.tolist()) if all_timestamps is not None else set()

    for i, frame in enumerate(frames_):
        actual_frame_num = int(output_timestamps[i]) if output_timestamps is not None else i + 1
        fi        = min(actual_frame_num - 1, len(tracked_detections) - 1)
        is_anchor = actual_frame_num in anchor_set

        # Skeleton via OpenCV
        if fi >= 0 and len(pose_results[fi].get('keypoints', [])) > 0:
            frame = draw_skeleton_opencv(
                frame,
                pose_results[fi]['keypoints'],
                pose_results[fi]['keypoint_scores'],
                kpt_thr=0.3
            )

        # Bbox + Track ID + Label — identik dengan versi HRNet
        if tracked_detections and fi >= 0:
            for t_idx, trk in enumerate(tracked_detections[fi]):
                x1, y1, x2, y2 = [int(v) for v in trk['bbox']]
                tid = trk['track_id']
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 200), 2)
                cv2.putText(frame, f'ID:{tid}',
                            (x1, max(0, y1 - 8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 200), 1)

                if is_anchor and annotations and i < len(annotations):
                    ann_frame = annotations[i]
                    if ann_frame and t_idx < len(ann_frame):
                        ann   = ann_frame[t_idx]
                        label = ann[1]
                        for kl, lb in enumerate(label):
                            if kl >= max_num:
                                break
                            text     = abbrev(lb)
                            location = (x1, y1 + 20 + kl * 20)
                            tw, th   = cv2.getTextSize(text, FONTFACE, 0.6, 1)[0]
                            cv2.rectangle(frame,
                                          (location[0], location[1] - th - 2),
                                          (location[0] + tw, location[1] + 2),
                                          plate[kl + 1], -1)
                            fc = (255, 0, 0) if lb in bahaya else (255, 255, 255)
                            cv2.putText(frame, text, location,
                                        FONTFACE, 0.6, fc, 1, LINETYPE)

        if is_anchor:
            frame_text  = f'Frame: {actual_frame_num}'
            (tw, th), _ = cv2.getTextSize(frame_text, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2)
            cv2.rectangle(frame, (8, 8), (18 + tw, 18 + th + 8), (0, 0, 0), -1)
            cv2.putText(frame, frame_text, (12, 12 + th),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 1, cv2.LINE_AA)

        frames_[i] = frame

    return frames_


## 6. Main Pipeline (Batch)

Output tiap video: `.csv` (berisi `annotate!,csv_id`) + `.pkl` + video anotasi.

**Setelah cell ini selesai:**
Buka setiap video output di `OUTPUT_DIR`, tonton, lalu isi kolom label
di `.csv` yang ada di `PKL_DIR` — sama persis seperti pipeline lama.
Setelah semua `.csv` diisi, lanjut ke Section 7.


In [9]:
video_list = sorted(
    glob.glob(os.path.join(INPUT_DIR, '**/*.avi'), recursive=True) +
    glob.glob(os.path.join(INPUT_DIR, '**/*.mp4'), recursive=True)
)
print(f'Total video ditemukan: {len(video_list)}')
for i, v in enumerate(video_list):
    base   = os.path.splitext(os.path.basename(v))[0]
    status = '✓' if os.path.exists(os.path.join(OUTPUT_DIR, f'{base}_out.mp4')) else '○'
    print(f'  [{i+1:02d}] {status} {os.path.basename(v)}')


Total video ditemukan: 1
  [01] ○ TaekwondoKicks.mp4


In [10]:
total = len(video_list)
success = skipped = failed = 0

for video_idx, video in enumerate(video_list):
    base_name    = os.path.splitext(os.path.basename(video))[0]
    ann_filename = os.path.join(PKL_DIR,    f'{base_name}.csv')
    pkl_filename = os.path.join(PKL_DIR,    f'{base_name}.pkl')
    out_filename = os.path.join(OUTPUT_DIR, f'{base_name}_out.mp4')

    print(f'\n{"="*55}')
    print(f'[{video_idx+1}/{total}] {base_name}')
    print(f'{"="*55}')

    if os.path.exists(out_filename):
        print('  ⏭  Skip — sudah diproses.')
        skipped += 1
        continue

    try:
        # STEP 1: Ekstrak frame
        print('  [1/5] Ekstrak frame...')
        tmp_dir = tempfile.TemporaryDirectory()
        frame_paths, original_frames = frame_extract(video, 480, out_dir=tmp_dir.name)
        num_frame = len(frame_paths)
        h, w, _   = original_frames[0].shape
        del original_frames; gc.collect()
        print(f'        {num_frame} frame | {w}x{h}')

        # STEP 2: YOLO11-pose + BoT-SORT (menggantikan Step 2+3 lama)
        print('  [2/5] YOLO11-pose + BoT-SORT...')
        tracked_detections, pose_results = run_detection_pose_yolo11(
            frame_paths, model_pose, det_score_thr,
            reid_weights=BOTSORT_REID_WEIGHTS,
            track_device=BOTSORT_DEVICE,
            half=BOTSORT_HALF
        )
        avg_p = sum(len(f) for f in tracked_detections) / max(len(tracked_detections), 1)
        print(f'        Rata-rata {avg_p:.1f} orang per frame')
        torch.cuda.empty_cache()

        # STEP 3: Clip extraction
        print('  [3/5] Clip extraction...')
        timestamps, stdet_preds, skeleton_datasets = skeleton_based_stdet_botsort(
            predict_stepsize, video,
            tracked_detections, pose_results,
            num_frame, predict_stepsize, 1, h, w
        )
        print(f'        {len(skeleton_datasets)} clip dihasilkan')

        # STEP 4: Simpan CSV + PKL
        # CSV berisi baris "annotate!,csv_id" — diisi label manual setelah ini
        print('  [4/5] Simpan CSV + PKL...')
        anno = ''
        for clip in stdet_preds:
            if clip is None:
                continue
            for person_attr in clip:
                anno += f'{person_attr[0][0]},{person_attr[0][1]:.3f}\n'
        with open(ann_filename, 'w') as f:
            f.write(anno)
        mmengine.dump(skeleton_datasets, pkl_filename)
        print(f'        CSV : {ann_filename}')
        print(f'        PKL : {pkl_filename}')

        # STEP 5: Render video anotasi untuk panduan labeling
        print('  [5/5] Render video anotasi...')
        anchor_anno_map = {}
        for timestamp, prediction in zip(timestamps, stdet_preds):
            if prediction is None:
                continue
            frame_anno = []
            for person_pred in prediction:
                if not person_pred:
                    continue
                label_str = person_pred[0][0]
                csv_id    = person_pred[0][1]
                frame_anno.append((
                    np.array([0, 0, 1, 1], dtype=np.float32),
                    [f'{label_str}: {csv_id:.3f}'],
                    [1.0]
                ))
            anchor_anno_map[int(timestamp)] = frame_anno

        output_timestamps = np.arange(1, num_frame + 1, dtype=np.int64)
        annotations_all   = [anchor_anno_map.get(int(ts), None) for ts in output_timestamps]
        frames_all        = [cv2.imread(frame_paths[min(ts-1, len(frame_paths)-1)])
                             for ts in output_timestamps]

        vis_frames = visualize_yolo11(
            frames_all, annotations_all, pose_results, None,
            tracked_detections=tracked_detections,
            output_timestamps=output_timestamps,
            all_timestamps=timestamps,
            predict_stepsize=predict_stepsize
        )

        vid = mpy.ImageSequenceClip(vis_frames, fps=output_fps)
        vid.write_videofile(out_filename, logger=None)
        print(f'        Video: {out_filename}')

        tmp_dir.cleanup()
        del frames_all, vis_frames, pose_results
        del tracked_detections, skeleton_datasets
        del timestamps, stdet_preds, anchor_anno_map, annotations_all
        gc.collect(); torch.cuda.empty_cache()

        success += 1
        print('  ✓ Selesai!')

    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        import traceback; traceback.print_exc()
        try: tmp_dir.cleanup()
        except: pass
        gc.collect(); torch.cuda.empty_cache()
        failed += 1

print(f'\n{"="*55}')
print(f'SELESAI  Berhasil:{success}  Skip:{skipped}  Gagal:{failed}')
print(f'{"="*55}')
print(f'→ Buka tiap video di {OUTPUT_DIR}/')
print(f'→ Isi label di tiap .csv di {PKL_DIR}/')
print(f'→ Setelah semua .csv diisi, lanjut ke Section 7.')



[1/1] TaekwondoKicks
  [1/5] Ekstrak frame...


2026-05-29 20:34:39.178 | INFO     | boxmot.utils.torch_utils:select_device:52 - Yolo Tracking v10.0.77 🚀 Python-3.8.18 torch-2.2.1
CUDA:0 (NVIDIA GeForce RTX 3090, 24252MiB)
2026-05-29 20:34:39.221 | SUCCESS  | boxmot.appearance.reid_model_factory:load_pretrained_weights:183 - Loaded pretrained weights from 01_asset_tools/models/osnet_x0_25_msmt17.pt


        5974 frame | 853x480
  [2/5] YOLO11-pose + BoT-SORT...
Running YOLO11-pose + BoT-SORT tracking...
[>>>>>>>>>>>>>>>           ] 3495/5974, 25.2 task/s, elapsed: 138s, ETA:    98s

2026-05-29 20:36:57.843 | WARNING  | boxmot.motion.cmc.sof:apply:142 - Affine matrix could not be generated: OpenCV(4.9.0) /io/opencv/modules/calib3d/src/ptsetreg.cpp:176: error: (-215:Assertion failed) count >= 0 && count2 == count in function 'run'



[>>>>>>>>>>>>>>>>>>>>>>>>>>] 5974/5974, 26.0 task/s, elapsed: 230s, ETA:     0s
Selesai. Total frame: 5974
        Rata-rata 4.1 orang per frame
  [3/5] Clip extraction...
Building track→pose index...
Extracting clips per track_id...
[>>>>>>>>>>>>>>>>>>>>>>>>>>] 1493/1493, 2340.4 task/s, elapsed: 1s, ETA:     0s        6093 clip dihasilkan
  [4/5] Simpan CSV + PKL...
        CSV : 03_dataset/data_in/2_csv/TaekwondoKicks.csv
        PKL : 03_dataset/data_in/2_csv/TaekwondoKicks.pkl
  [5/5] Render video anotasi...
        Video: 03_dataset/data_out/2_out/TaekwondoKicks_out.mp4
  ✓ Selesai!

SELESAI  Berhasil:1  Skip:0  Gagal:0
→ Buka tiap video di 03_dataset/data_out/2_out/
→ Isi label di tiap .csv di 03_dataset/data_in/2_csv/
→ Setelah semua .csv diisi, lanjut ke Section 7.


## 7. Add Label ke Dataset

Edit file `.csv` dulu (ganti `annotate!` → nama label aksi), 
lalu jalankan cell berikut. **Sama persis dengan pipeline lama.**


In [ ]:
import csv as csv_module

def load_label_map(file_path):
    lines = open(file_path).readlines()
    return {x[1]: int(x[0]) for x in [l.strip().split(': ') for l in lines]}

stdet_label_map = load_label_map(label_map_stdet)
print('Label map:', stdet_label_map)


In [ ]:
# Jalankan untuk semua video sekaligus
all_custom_dataset = []
LABELED_DIR        = 'dataset/data_in/pkl_labeled_yolo11'
os.makedirs(LABELED_DIR, exist_ok=True)

csv_files = sorted(glob.glob(os.path.join(PKL_DIR, '*.csv')))
print(f'Total file CSV: {len(csv_files)}\n')

for csv_file in csv_files:
    base     = os.path.splitext(os.path.basename(csv_file))[0]
    pkl_file = os.path.join(PKL_DIR, f'{base}.pkl')

    if not os.path.exists(pkl_file):
        print(f'  SKIP (pkl tidak ada): {base}')
        continue

    # Load anotasi dari CSV
    custom_annos = []
    with open(csv_file, newline='') as f:
        for row in csv_module.reader(f):
            if not row or row[0] in ('none', 'annotate!', ''):
                continue
            if row[0] not in stdet_label_map:
                print(f'  ⚠ Label tidak dikenal: {row[0]} — dilewati')
                continue
            custom_annos.append([float(row[1]), stdet_label_map[row[0]]])

    if not custom_annos:
        print(f'  SKIP (belum ada label): {base}')
        continue

    # Match csv_id ke pkl
    skeleton_datasets = mmengine.load(pkl_file)
    custom_dataset    = []

    for idx, ann in enumerate(custom_annos):
        csv_id_target = ann[0]
        for data in skeleton_datasets:
            if abs(data.get('csv_id', -1) - csv_id_target) < 1e-6:
                labeled               = cp.deepcopy(data)
                labeled['frame_dir'] += f'_{idx}'
                labeled['label']      = int(ann[1])
                labeled['clip_len']   = data['total_frames']  # tambah clip_len
                custom_dataset.append(labeled)
                break

    # Simpan pkl berlabel per video
    out_path = os.path.join(LABELED_DIR, f'{base}_labeled.pkl')
    mmengine.dump(custom_dataset, out_path)
    all_custom_dataset.extend(custom_dataset)
    print(f'  {base}: {len(custom_dataset)} clip berlabel')

print(f'\nTotal semua clip berlabel: {len(all_custom_dataset)}')


## 8. Combine PKL + Split Train/Val

In [ ]:
# Gabungkan data YOLO11-pose baru dengan data kating
# Data kating dipakai ulang as-is — tidak perlu re-ekstraksi
KATING_PKL  = 'dataset/pkl_fin_new/ciis_gab2_fin.pkl'  # pkl gabungan lama
OUTPUT_PKL  = 'dataset/pkl_fin_new/ciis_yolo11_fin.pkl'
SPLIT_RATIO = 0.8  # 80% train, 20% val

kating_data        = mmengine.load(KATING_PKL)
kating_annotations = kating_data['annotations']

print(f'Data kating : {len(kating_annotations)} clip')
print(f'Data baru   : {len(all_custom_dataset)} clip')
print(f'Total       : {len(kating_annotations) + len(all_custom_dataset)} clip')


In [ ]:
combined = kating_annotations + all_custom_dataset

custom_datasets = dict(
    split=dict(xsub_train=[], xsub_val=[], xview_train=[], xview_val=[]),
    annotations=[]
)

for i, data in enumerate(combined):
    custom_datasets['annotations'].append(data)
    split_key = 'train' if (i % 10) < (SPLIT_RATIO * 10) else 'val'
    custom_datasets['split'][f'xsub_{split_key}'].append(data['frame_dir'])
    custom_datasets['split'][f'xview_{split_key}'].append(data['frame_dir'])

mmengine.dump(custom_datasets, OUTPUT_PKL)

n_tr = len(custom_datasets['split']['xsub_train'])
n_va = len(custom_datasets['split']['xsub_val'])
print(f'\nDisimpan : {OUTPUT_PKL}')
print(f'Train    : {n_tr}')
print(f'Val      : {n_va}')
print(f'Total    : {n_tr + n_va}')
print()
print('Selanjutnya → buat ciis_7_yolo11.py dengan ann_file diganti ke:')
print(f'              {OUTPUT_PKL}')
print('             lalu training ulang untuk komparasi.')
